# 05. Content-Based Recommendation & Constraint-Based Combination System
**Seksi 9, 10, & 11 (Eksperimen E6 & E7)** pada `instruction.md`

### Fitur Sistem:
1. **Content-Based Recommendation**: Memberikan alternatif barang serupa ketika barang yang dicari kosong/habis.
2. **Constraint-Based Combination**: Menyusun bundle kombinasi barang untuk kebutuhan aktivitas asrama (Dekorasi, Acara, Olahraga, Maintenance) lengkap dengan validasi ketersediaan stok aktual.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

inv = pd.read_csv('../data/processed/inventory_catalog.csv')
rules = pd.read_csv('../data/processed/combination_rules.csv')

inv['doc'] = inv['category'] + ': ' + inv['item_name'] + '. ' + inv['description'].fillna('')
sbert = SentenceTransformer('all-MiniLM-L6-v2')
inv_vectors = sbert.encode(inv['doc'].tolist(), normalize_embeddings=True)
item_sim_matrix = cosine_similarity(inv_vectors, inv_vectors)

print(f'Katalog Inventaris: {len(inv)} item')
print(f'Aturan Kombinasi : {len(rules)} aturan')

In [ ]:
def get_alternatives(item_name, top_n=3):
    matches = inv[inv['item_name'].str.contains(item_name, case=False, na=False)]
    if matches.empty:
        return f"Barang '{item_name}' tidak ditemukan."
    
    idx = matches.index[0]
    target_row = inv.iloc[idx]
    sim_scores = item_sim_matrix[idx]
    
    top_idx = np.argsort(sim_scores)[::-1]
    top_idx = [i for i in top_idx if i != idx][:top_n]
    
    res = []
    for rank, i in enumerate(top_idx, 1):
        r = inv.iloc[i]
        res.append({
            'Alternatif': rank,
            'Nama Barang': r['item_name'],
            'Kategori': r['category'],
            'Skor Kecocokan': round(float(sim_scores[i]), 4),
            'Stok': f"{r['quantity']} {r['unit']}",
            'Lokasi': r['location']
        })
    return pd.DataFrame(res)

# Rekomendasi alternatif jika barang target dicari
sample_item = inv.iloc[0]['item_name'][:20]
print(f'Mencari alternatif untuk barang serupa: {sample_item}')
get_alternatives(sample_item, top_n=3)

In [ ]:
def recommend_activity_bundle(need_name):
    matched_rules = rules[rules['need'].str.lower() == need_name.lower()]
    if matched_rules.empty:
        return f"Kebutuhan tidak terdaftar. Pilihan: {rules['need'].unique().tolist()}"
    
    bundle = []
    for _, rule in matched_rules.iterrows():
        target_cat = rule['target_category']
        req_qty = rule['default_qty']
        
        # Ambil barang kandidat terbaik dalam kategori tersebut dengan stok terbesar
        candidates = inv[inv['category'] == target_cat].sort_values(by='quantity', ascending=False)
        if not candidates.empty:
            selected = candidates.iloc[0]
            curr_stock = selected['quantity']
            status = 'Tersedia' if curr_stock >= req_qty else 'Stok Menipis'
            
            bundle.append({
                'Peran / Komponen': rule['role_required'],
                'Barang Rekomendasi': selected['item_name'],
                'Kategori': selected['category'],
                'Jumlah Dibutuhkan': req_qty,
                'Stok Gudang': curr_stock,
                'Lokasi Simpan': selected['location'],
                'Status Kesiapan': status,
                'Batasan': rule['constraints']
            })
    return pd.DataFrame(bundle)

print('=== PAKET KOMBINASI KEGIATAN: Dekorasi Panggung Acara ===')
recommend_activity_bundle('Dekorasi Panggung Acara')